# SE-BRL Feature Extraction

## Purpose

This notebook derives the modality-aware Social Engineering Behavior
Representation Layer (SE-BRL) from the fixed preprocessing outputs produced by
`03_preprocessing.ipynb`.

The SE-BRL representation consists of four behavioral dimensions supported by
twelve subordinate indicators:

### Pressure and Threat
- Urgency
- Fear or Threat
- Scarcity

### Lure and Attention
- Curiosity
- Reward or Lure

### Trust and Identity
- Authority
- Impersonation
- Brand Exploitation
- Confidentiality or Isolation

### Requested Action and Consequence
- Call-to-Action
- Credential or Sensitive-Data Request
- Financial-Action Request

Feature extraction is deterministic and does not introduce researcher-assigned
behavior labels.

Each behavioral dimension is represented using one of three evidence states:

- `Supported` — qualifying evidence is present;
- `Absent` — the dimension is assessable but qualifying evidence is not found;
- `Unavailable` — the required modality or evidence field is not assessable.

Exact evidence references are retained separately for explainability and
validation but are not directly used as predictive-vector inputs.

## Modality Rules

Email observations may be assessed using available subject/body content,
sender or identity relations, links, formatting, and related message fields.

Webpage observations may be assessed using available text, HTML/DOM structure,
forms, links, brand/domain relations, and other supplied page evidence.

Standalone URL or engineered URL-feature observations do not provide sufficient
content for language-dependent SE-BRL behavioral inference. Their corresponding
behavioral dimensions are therefore marked `Unavailable` rather than `Absent`.

## Inputs

Fixed preprocessing outputs:

`experiments/data/processed/preprocessed/`

Partition metadata:

`experiments/results/preprocessing/partitions/`

## Outputs

The notebook will produce:

- deterministic indicator-level evidence;
- dimension-level SE-BRL states;
- SE-BRL behavioral vectors;
- modality availability masks; and
- evidence records for downstream validation and explanation.

The fixed observation, group, partition, and cross-validation assignments from
`03_preprocessing.ipynb` are preserved unchanged.

## 1. Environment and Paths

Feature extraction operates only on the fixed preprocessing outputs.

No partition assignments are regenerated and no target labels are modified in
this notebook.

In [1]:
from pathlib import Path
import hashlib
import json
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

In [2]:
cwd = Path.cwd().resolve()

REPO_ROOT = next(
    (
        p for p in [cwd, *cwd.parents]
        if (p / ".git").exists()
        and (p / "experiments").exists()
    ),
    None
)

if REPO_ROOT is None:
    raise RuntimeError("Could not locate repository root.")

PREPROCESSED_DIR = (
    REPO_ROOT
    / "experiments"
    / "data"
    / "processed"
    / "preprocessed"
)

PREPROCESSING_RESULTS_DIR = (
    REPO_ROOT
    / "experiments"
    / "results"
    / "preprocessing"
)

FEATURE_RESULTS_DIR = (
    REPO_ROOT
    / "experiments"
    / "results"
    / "feature_extraction"
)

FEATURE_DATA_DIR = (
    REPO_ROOT
    / "experiments"
    / "data"
    / "processed"
    / "features"
)

FEATURE_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FEATURE_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Repository root:", REPO_ROOT)
print("Preprocessed input:", PREPROCESSED_DIR)
print("Preprocessing results:", PREPROCESSING_RESULTS_DIR)
print("Feature results:", FEATURE_RESULTS_DIR)
print("Feature data:", FEATURE_DATA_DIR)

Repository root: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL
Preprocessed input: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL/experiments/data/processed/preprocessed
Preprocessing results: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL/experiments/results/preprocessing
Feature results: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL/experiments/results/feature_extraction
Feature data: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL/experiments/data/processed/features


In [3]:
print("Preprocessed input exists:", PREPROCESSED_DIR.exists())
print("Preprocessing results exist:", PREPROCESSING_RESULTS_DIR.exists())
print("Feature results exist:", FEATURE_RESULTS_DIR.exists())
print("Feature data exists:", FEATURE_DATA_DIR.exists())

Preprocessed input exists: True
Preprocessing results exist: True
Feature results exist: True
Feature data exists: True


## 2. Canonical SE-BRL Codebook

The SE-BRL representation is defined independently of any individual dataset.

Each behavioral indicator belongs to exactly one behavioral dimension, and each
dimension is assigned one of three evidence states:

- `Supported` — qualifying behavioral evidence is detected;
- `Absent` — the dimension is assessable but qualifying evidence is not detected;
- `Unavailable` — the modality does not provide sufficient evidence to assess
  the dimension.

The codebook defined in this section is used consistently across extraction,
validation, and downstream experiments.

In [4]:
SEBRL_CODEBOOK = {
    "pressure_threat": {
        "name": "Pressure / Threat",
        "indicators": {
            "urgency": "Urgency",
            "fear_threat": "Fear or Threat",
            "scarcity": "Scarcity",
        },
    },

    "lure_attention": {
        "name": "Lure / Attention",
        "indicators": {
            "curiosity": "Curiosity",
            "reward_lure": "Reward or Lure",
        },
    },

    "trust_identity": {
        "name": "Trust / Identity",
        "indicators": {
            "authority": "Authority",
            "impersonation": "Impersonation",
            "brand_exploitation": "Brand Exploitation",
            "confidentiality_isolation": "Confidentiality / Isolation",
        },
    },

    "requested_action_consequence": {
        "name": "Requested Action / Consequence",
        "indicators": {
            "call_to_action": "Call-to-Action",
            "credential_sensitive_request":
                "Credential / Sensitive-Data Request",
            "financial_action_request":
                "Financial-Action Request",
        },
    },
}

In [5]:
codebook_rows = []

for dimension_id, dimension_info in SEBRL_CODEBOOK.items():
    for indicator_id, indicator_name in dimension_info["indicators"].items():
        codebook_rows.append({
            "dimension_id": dimension_id,
            "dimension_name": dimension_info["name"],
            "indicator_id": indicator_id,
            "indicator_name": indicator_name,
        })

sebrl_codebook_df = pd.DataFrame(codebook_rows)

print("Dimensions:", sebrl_codebook_df["dimension_id"].nunique())
print("Indicators:", len(sebrl_codebook_df))

sebrl_codebook_df

Dimensions: 4
Indicators: 12


,dimension_id,dimension_name,indicator_id,indicator_name
0,pressure_threat,Pressure / Threat,urgency,Urgency
1,pressure_threat,Pressure / Threat,fear_threat,Fear or Threat
2,pressure_threat,Pressure / Threat,scarcity,Scarcity
3,lure_attention,Lure / Attention,curiosity,Curiosity
4,lure_attention,Lure / Attention,reward_lure,Reward or Lure
5,trust_identity,Trust / Identity,authority,Authority
6,trust_identity,Trust / Identity,impersonation,Impersonation
7,trust_identity,Trust / Identity,brand_exploitation,Brand Exploitation
8,trust_identity,Trust / Identity,confidentiality_isolation,Confidentiality / Isolation
9,requested_action_consequence,Requested Action / Consequence,call_to_action,Call-to-Action


In [6]:
SEBRL_STATES = [
    "Supported",
    "Absent",
    "Unavailable",
]

print(SEBRL_STATES)

['Supported', 'Absent', 'Unavailable']


In [7]:
MODALITY_AVAILABILITY = {
    "email": {
        "pressure_threat": True,
        "lure_attention": True,
        "trust_identity": True,
        "requested_action_consequence": True,
    },

    "webpage": {
        "pressure_threat": True,
        "lure_attention": True,
        "trust_identity": True,
        "requested_action_consequence": True,
    },

    "url": {
        "pressure_threat": False,
        "lure_attention": False,
        "trust_identity": False,
        "requested_action_consequence": False,
    },
}

In [8]:
expected_dimensions = set(SEBRL_CODEBOOK.keys())

for modality, availability in MODALITY_AVAILABILITY.items():
    assert set(availability.keys()) == expected_dimensions

print("All modality availability rules cover all SE-BRL dimensions.")

All modality availability rules cover all SE-BRL dimensions.


### 2.1 Codebook Validation Result

The canonical SE-BRL codebook contains four behavioral dimensions and twelve
subordinate indicators.

All three supported modalities have explicit availability rules for every
dimension.

Email and webpage observations are behavior-assessable, while standalone URL
feature records are marked unavailable for behavioral inference because they do
not provide message or page-content evidence.

## 3. Load Fixed Preprocessed Inputs

Feature extraction begins from the fixed outputs produced by
`03_preprocessing.ipynb`.

The observation identifiers, leakage-prevention groups, development/calibration/
test partitions, and development cross-validation assignments are treated as
immutable metadata.

This notebook does not regenerate partitions or modify standardized targets.

In [9]:
FEATURE_INPUTS = {
    "Email":
        PREPROCESSED_DIR / "email_preprocessed.csv",

    "PhiUSIIL":
        PREPROCESSED_DIR / "phiusiil_preprocessed.csv",

    "UCI":
        PREPROCESSED_DIR / "uci_phishing_websites_preprocessed.csv",

    "ISCX":
        PREPROCESSED_DIR / "iscx_url2016_preprocessed.csv",
}

for name, path in FEATURE_INPUTS.items():
    print(f"{name:10s} {'OK' if path.exists() else 'MISSING'}")

Email      OK
PhiUSIIL   OK
UCI        OK
ISCX       OK


In [10]:
feature_data = {
    name: pd.read_csv(path)
    for name, path in FEATURE_INPUTS.items()
}

for name, df in feature_data.items():
    print(
        f"{name:10s} "
        f"{len(df):,} rows × {len(df.columns)} columns"
    )

Email      38,759 rows × 17 columns
PhiUSIIL   235,795 rows × 64 columns
UCI        5,849 rows × 39 columns
ISCX       14,823 rows × 88 columns


In [11]:
required_metadata = {
    "observation_id",
    "target",
    "group_id",
    "partition",
    "cv_fold",
}

for name, df in feature_data.items():
    missing = required_metadata - set(df.columns)

    print(
        f"{name}: "
        f"{'OK' if not missing else f'MISSING {missing}'}"
    )

Email: OK
PhiUSIIL: OK
UCI: OK
ISCX: OK


In [12]:
for name, df in feature_data.items():

    print(f"\n{name}")

    print(
        "Unique observation IDs:",
        df["observation_id"].nunique()
    )

    print(
        "Duplicate observation IDs:",
        df["observation_id"].duplicated().sum()
    )

    print(
        "Missing partitions:",
        df["partition"].isna().sum()
    )

    dev_missing_folds = df.loc[
        df["partition"] == "development",
        "cv_fold"
    ].isna().sum()

    nondev_with_folds = df.loc[
        df["partition"] != "development",
        "cv_fold"
    ].notna().sum()

    print(
        "Development rows missing CV fold:",
        dev_missing_folds
    )

    print(
        "Calibration/test rows with CV fold:",
        nondev_with_folds
    )


Email
Unique observation IDs: 38759
Duplicate observation IDs: 0
Missing partitions: 0
Development rows missing CV fold: 0
Calibration/test rows with CV fold: 0

PhiUSIIL
Unique observation IDs: 235795
Duplicate observation IDs: 0
Missing partitions: 0
Development rows missing CV fold: 0
Calibration/test rows with CV fold: 0

UCI
Unique observation IDs: 5849
Duplicate observation IDs: 0
Missing partitions: 0
Development rows missing CV fold: 0
Calibration/test rows with CV fold: 0

ISCX
Unique observation IDs: 14823
Duplicate observation IDs: 0
Missing partitions: 0
Development rows missing CV fold: 0
Calibration/test rows with CV fold: 0


## 4. Modality-Specific Evidence Availability

SE-BRL feature extraction depends on the evidence exposed by each modality.

The available fields differ substantially across the email, webpage, and URL
datasets. Behavioral inference is therefore restricted to evidence that is
actually present in the corresponding source representation.

This section records which fields are available for:

- textual content;
- sender or identity information;
- links and URLs;
- domain or brand information;
- structural or form-related webpage evidence; and
- engineered URL-only features.

Fields that are unavailable for a modality are not treated as negative evidence.
They result in `Unavailable` states where the corresponding behavioral
dimension cannot be assessed.

In [13]:
for name, df in feature_data.items():
    print(f"\n{name}")
    print(df.columns.tolist())


Email
['sender', 'receiver', 'date', 'subject', 'body', 'urls', 'label', 'source_dataset', 'target', 'source_row_id', 'observation_id', 'normalized_body', 'body_hash', 'normalized_sender', 'group_id', 'partition', 'cv_fold']

PhiUSIIL
['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'Ha

In [14]:
def has_any_column(df, candidates):
    return any(
        col in df.columns
        for col in candidates
    )

In [15]:
evidence_availability = []

for name, df in feature_data.items():

    if name == "Email":
        modality = "email"

    elif name == "PhiUSIIL":
        modality = "webpage"

    else:
        modality = "url"

    evidence_availability.append({
        "dataset": name,
        "modality": modality,

        "text_content":
            has_any_column(
                df,
                ["subject", "body", "Title"]
            ),

        "sender_identity":
            has_any_column(
                df,
                ["sender"]
            ),

        "url_or_link":
            has_any_column(
                df,
                ["urls", "URL"]
            ),

        "domain_identity":
            has_any_column(
                df,
                ["Domain", "TLD"]
            ),

        "structured_web_features":
            (
                name == "PhiUSIIL"
            ),

        "engineered_url_features":
            (
                name in {
                    "UCI",
                    "ISCX"
                }
            ),
    })

evidence_availability_df = pd.DataFrame(
    evidence_availability
)

evidence_availability_df

,dataset,modality,text_content,sender_identity,url_or_link,domain_identity,structured_web_features,engineered_url_features
0,Email,email,True,True,True,False,False,False
1,PhiUSIIL,webpage,True,False,True,True,True,False
2,UCI,url,False,False,False,False,False,True
3,ISCX,url,False,False,False,False,False,True


### 4.1 Evidence Availability Interpretation

Email provides message content and, for some sources, sender and link-related
metadata. These fields support direct behavioral assessment.

PhiUSIIL provides URL/domain information together with webpage-oriented
features and title-related evidence. Its available fields support webpage
behavioral assessment, although individual indicators remain constrained by the
specific evidence represented in the dataset.

UCI and ISCX provide engineered URL or webpage-characteristic features without
the underlying message or webpage content required for direct social-engineering
behavior inference.

Accordingly, UCI and ISCX retain `Unavailable` behavioral states for
language-dependent SE-BRL dimensions rather than treating missing behavioral
evidence as `Absent`.

## 5. Deterministic Indicator Extraction Rules

SE-BRL indicator extraction is rule-based and deterministic.

Each indicator is defined using explicit lexical, structural, or identity-based
evidence that can be reproduced automatically from the available fields.

The extraction rules do not assign behavioral labels manually and do not depend
on researcher judgment at runtime.

Where a modality does not expose the evidence required for an indicator, the
corresponding dimension may be marked `Unavailable` rather than `Absent`.

In [16]:
SEBRL_RULES = {
    "urgency": {
        "type": "lexical",
        "patterns": [
            r"\burgent\b",
            r"\bimmediately\b",
            r"\basap\b",
            r"\bact now\b",
            r"\bright away\b",
            r"\bwithin \d+ (?:hour|hours|minute|minutes)\b",
            r"\bdeadline\b",
            r"\bexpires?\b",
        ],
    },

    "fear_threat": {
        "type": "lexical",
        "patterns": [
            r"\bsuspended\b",
            r"\bterminated\b",
            r"\blocked\b",
            r"\bunauthorized\b",
            r"\bsecurity alert\b",
            r"\bcompromised\b",
            r"\bpenalty\b",
            r"\blegal action\b",
            r"\baccount closure\b",
        ],
    },

    "scarcity": {
        "type": "lexical",
        "patterns": [
            r"\blimited time\b",
            r"\blimited offer\b",
            r"\bwhile supplies last\b",
            r"\bonly \d+ left\b",
            r"\blast chance\b",
            r"\bexclusive offer\b",
        ],
    },

    "curiosity": {
        "type": "lexical",
        "patterns": [
            r"\bsee what happened\b",
            r"\bcheck this out\b",
            r"\byou won't believe\b",
            r"\bimportant message\b",
            r"\bprivate message\b",
            r"\bsurprise\b",
        ],
    },

    "reward_lure": {
        "type": "lexical",
        "patterns": [
            r"\bwon\b",
            r"\bwinner\b",
            r"\bprize\b",
            r"\breward\b",
            r"\bbonus\b",
            r"\bgift\b",
            r"\bfree\b",
            r"\bcashback\b",
            r"\bclaim\b",
        ],
    },

    "authority": {
        "type": "lexical",
        "patterns": [
            r"\badministrator\b",
            r"\badmin\b",
            r"\bmanagement\b",
            r"\bsecurity team\b",
            r"\bit department\b",
            r"\bgovernment\b",
            r"\bbank\b",
            r"\bmanager\b",
            r"\bceo\b",
        ],
    },

    "impersonation": {
        "type": "identity",
        "patterns": [],
    },

    "brand_exploitation": {
        "type": "identity",
        "patterns": [],
    },

    "confidentiality_isolation": {
        "type": "lexical",
        "patterns": [
            r"\bconfidential\b",
            r"\bdo not share\b",
            r"\bkeep this private\b",
            r"\bsecret\b",
            r"\bdo not tell\b",
            r"\bbetween us\b",
        ],
    },

    "call_to_action": {
        "type": "lexical",
        "patterns": [
            r"\bclick\b",
            r"\bopen\b",
            r"\bvisit\b",
            r"\bdownload\b",
            r"\bverify\b",
            r"\bconfirm\b",
            r"\bupdate\b",
            r"\bsign in\b",
            r"\blog in\b",
            r"\breply\b",
        ],
    },

    "credential_sensitive_request": {
        "type": "lexical",
        "patterns": [
            r"\bpassword\b",
            r"\busername\b",
            r"\blogin credentials\b",
            r"\bverification code\b",
            r"\botp\b",
            r"\bpin\b",
            r"\bssn\b",
            r"\bsocial security\b",
            r"\bpersonal information\b",
            r"\baccount details\b",
        ],
    },

    "financial_action_request": {
        "type": "lexical",
        "patterns": [
            r"\btransfer\b",
            r"\bpayment\b",
            r"\bpay\b",
            r"\binvoice\b",
            r"\bwire\b",
            r"\bbank transfer\b",
            r"\bcredit card\b",
            r"\bdebit card\b",
            r"\bfee\b",
            r"\bdeposit\b",
        ],
    },
}

In [17]:
expected_indicators = set(
    sebrl_codebook_df["indicator_id"]
)

defined_indicators = set(
    SEBRL_RULES.keys()
)

print(
    "Missing rules:",
    expected_indicators - defined_indicators
)

print(
    "Unexpected rules:",
    defined_indicators - expected_indicators
)

assert defined_indicators == expected_indicators

print("All 12 SE-BRL indicators have rule definitions.")

Missing rules: set()
Unexpected rules: set()
All 12 SE-BRL indicators have rule definitions.


### 5.1 Lexical Matching and Evidence Capture

Lexical indicators are evaluated using deterministic regular-expression rules.

For each indicator, the extraction process records:

- whether qualifying evidence is present; and
- the exact matched text fragments that support the indicator.

Evidence is preserved for explainability and validation. Matched phrases are
not manually edited or reinterpreted.

In [18]:
def normalize_match_text(value):
    if pd.isna(value):
        return ""

    value = str(value)

    # Normalize whitespace while preserving original wording
    value = re.sub(r"\s+", " ", value)

    return value.strip()


def extract_pattern_evidence(text, patterns):
    text = normalize_match_text(text)

    if not text:
        return []

    evidence = []

    for pattern in patterns:
        matches = re.finditer(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        for match in matches:
            matched_text = match.group(0).strip()

            if matched_text:
                evidence.append(matched_text)

    # Preserve order while removing duplicates
    seen = set()
    unique_evidence = []

    for item in evidence:
        key = item.lower()

        if key not in seen:
            seen.add(key)
            unique_evidence.append(item)

    return unique_evidence

In [19]:
def evaluate_lexical_indicator(text, indicator_id):
    rule = SEBRL_RULES[indicator_id]

    if rule["type"] != "lexical":
        raise ValueError(
            f"{indicator_id} is not a lexical indicator."
        )

    evidence = extract_pattern_evidence(
        text,
        rule["patterns"]
    )

    return {
        "supported": len(evidence) > 0,
        "evidence": evidence
    }

In [20]:
test_examples = [
    "Your account will be suspended. Verify immediately.",
    "Congratulations! You won a free prize. Claim it now.",
    "Please send your password and verification code.",
    "Meeting moved to Tuesday afternoon."
]

for text in test_examples:

    print("\nTEXT:", text)

    for indicator_id, rule in SEBRL_RULES.items():

        if rule["type"] != "lexical":
            continue

        result = evaluate_lexical_indicator(
            text,
            indicator_id
        )

        if result["supported"]:
            print(
                indicator_id,
                "→",
                result["evidence"]
            )


TEXT: Your account will be suspended. Verify immediately.
urgency → ['immediately']
fear_threat → ['suspended']
call_to_action → ['Verify']

TEXT: Congratulations! You won a free prize. Claim it now.
reward_lure → ['won', 'prize', 'free', 'Claim']

TEXT: Please send your password and verification code.
credential_sensitive_request → ['password', 'verification code']

TEXT: Meeting moved to Tuesday afternoon.


## 6. Canonical Evidence Text Construction

Lexical SE-BRL indicators are evaluated only against modality-appropriate text
fields.

For email observations, the canonical textual evidence consists of the subject
and body content.

For PhiUSIIL webpage observations, the available natural-language text field is
the page title. URL and domain fields are preserved separately for identity,
brand, and structural evidence rather than being merged into the lexical text
representation.

UCI and ISCX do not expose natural-language message or webpage content and
therefore do not receive lexical behavioral extraction.

In [22]:
email_features = feature_data["Email"].copy()

email_features["evidence_text"] = (
    email_features["subject"]
    .fillna("")
    .astype(str)
    .str.cat(
        email_features["body"]
        .fillna("")
        .astype(str),
        sep=" "
    )
    .map(normalize_match_text)
)

phiusiil_features = feature_data["PhiUSIIL"].copy()

phiusiil_features["evidence_text"] = (
    phiusiil_features["Title"]
    .fillna("")
    .astype(str)
    .map(normalize_match_text)
)

uci_features = feature_data["UCI"].copy()
iscx_features = feature_data["ISCX"].copy()

In [23]:
print(
    "Email rows with non-empty evidence text:",
    (email_features["evidence_text"] != "").sum(),
    "/",
    len(email_features)
)

print(
    "PhiUSIIL rows with non-empty evidence text:",
    (phiusiil_features["evidence_text"] != "").sum(),
    "/",
    len(phiusiil_features)
)

Email rows with non-empty evidence text: 38759 / 38759
PhiUSIIL rows with non-empty evidence text: 235795 / 235795


In [24]:
LEXICAL_INDICATORS = [
    indicator_id
    for indicator_id, rule in SEBRL_RULES.items()
    if rule["type"] == "lexical"
]

print("Lexical indicators:", len(LEXICAL_INDICATORS))
print(LEXICAL_INDICATORS)

Lexical indicators: 10
['urgency', 'fear_threat', 'scarcity', 'curiosity', 'reward_lure', 'authority', 'confidentiality_isolation', 'call_to_action', 'credential_sensitive_request', 'financial_action_request']


### 6.1 Apply Lexical Indicator Extraction

The ten lexical SE-BRL indicators are applied only to modalities with
natural-language evidence.

For each indicator, two outputs are produced:

- a boolean support flag; and
- the exact matched evidence fragments.

Email extraction uses the combined subject and body text.

PhiUSIIL extraction uses the available webpage title field.

UCI and ISCX remain unavailable for lexical behavioral extraction because they
do not expose natural-language message or page content.

In [26]:
def apply_lexical_rules(df, text_column, prefix=""):
    df = df.copy()

    for indicator_id in LEXICAL_INDICATORS:

        support_col = f"{prefix}{indicator_id}_supported"
        evidence_col = f"{prefix}{indicator_id}_evidence"

        results = df[text_column].map(
            lambda text: evaluate_lexical_indicator(
                text,
                indicator_id
            )
        )

        df[support_col] = results.map(
            lambda x: x["supported"]
        )

        df[evidence_col] = results.map(
            lambda x: json.dumps(
                x["evidence"],
                ensure_ascii=False
            )
        )

    return df

In [30]:
email_features = apply_lexical_rules(
    email_features,
    text_column="evidence_text"
)

In [31]:
phiusiil_features = apply_lexical_rules(
    phiusiil_features,
    text_column="evidence_text"
)

In [32]:
email_support_counts = {
    indicator_id:
        int(
            email_features[
                f"{indicator_id}_supported"
            ].sum()
        )
    for indicator_id in LEXICAL_INDICATORS
}

phiusiil_support_counts = {
    indicator_id:
        int(
            phiusiil_features[
                f"{indicator_id}_supported"
            ].sum()
        )
    for indicator_id in LEXICAL_INDICATORS
}

print("Email support counts:")
print(email_support_counts)

print("\nPhiUSIIL support counts:")
print(phiusiil_support_counts)

Email support counts:
{'urgency': 3224, 'fear_threat': 850, 'scarcity': 122, 'curiosity': 207, 'reward_lure': 5315, 'authority': 5748, 'confidentiality_isolation': 1054, 'call_to_action': 11820, 'credential_sensitive_request': 1254, 'financial_action_request': 3208}

PhiUSIIL support counts:
{'urgency': 29, 'fear_threat': 0, 'scarcity': 0, 'curiosity': 0, 'reward_lure': 67, 'authority': 96, 'confidentiality_isolation': 8, 'call_to_action': 371, 'credential_sensitive_request': 8, 'financial_action_request': 48}


In [33]:
lexical_support_summary = pd.DataFrame({
    "indicator_id": LEXICAL_INDICATORS,
    "email_supported": [
        email_support_counts[i]
        for i in LEXICAL_INDICATORS
    ],
    "phiusiil_supported": [
        phiusiil_support_counts[i]
        for i in LEXICAL_INDICATORS
    ]
})

lexical_support_summary

,indicator_id,email_supported,phiusiil_supported
0,urgency,3224,29
1,fear_threat,850,0
2,scarcity,122,0
3,curiosity,207,0
4,reward_lure,5315,67
5,authority,5748,96
6,confidentiality_isolation,1054,8
7,call_to_action,11820,371
8,credential_sensitive_request,1254,8
9,financial_action_request,3208,48


## 7. Identity and Structural Evidence Review

The remaining SE-BRL indicators `impersonation` and `brand_exploitation`
require identity-related evidence rather than simple lexical matching.

Before defining deterministic rules, the available email and webpage fields are
reviewed for evidence that can support:

- claimed identity;
- sender identity;
- domain identity;
- brand references;
- title/domain consistency; and
- webpage structural cues.

No identity-based indicator is assigned unless the available fields support a
reproducible rule.

In [34]:
email_identity_candidates = [
    col
    for col in email_features.columns
    if any(
        term in col.lower()
        for term in [
            "sender",
            "from",
            "reply",
            "url",
            "domain",
            "subject"
        ]
    )
]

print("Email identity-related columns:")
print(email_identity_candidates)

Email identity-related columns:
['sender', 'subject', 'urls', 'normalized_sender']


In [35]:
phiusiil_identity_candidates = [
    col
    for col in phiusiil_features.columns
    if any(
        term in col.lower()
        for term in [
            "url",
            "domain",
            "title",
            "brand",
            "login",
            "form",
            "password",
            "submit",
            "iframe",
            "redirect",
            "external",
            "link"
        ]
    )
]

print("PhiUSIIL identity / structural columns:")
print(phiusiil_identity_candidates)

PhiUSIIL identity / structural columns:
['URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'URLSimilarityIndex', 'URLCharProb', 'NoOfSubDomain', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSubmitButton', 'HasPasswordField', 'NoOfExternalRef', 'normalized_domain']


In [36]:
print("\nEmail:")
print(
    email_features[
        email_identity_candidates
    ].dtypes
)

print("\nPhiUSIIL:")
print(
    phiusiil_features[
        phiusiil_identity_candidates
    ].dtypes
)


Email:
sender                object
subject               object
urls                 float64
normalized_sender     object
dtype: object

PhiUSIIL:
URL                            object
URLLength                       int64
Domain                         object
DomainLength                    int64
IsDomainIP                      int64
URLSimilarityIndex            float64
URLCharProb                   float64
NoOfSubDomain                   int64
NoOfLettersInURL                int64
LetterRatioInURL              float64
NoOfDegitsInURL                 int64
DegitRatioInURL               float64
NoOfEqualsInURL                 int64
NoOfQMarkInURL                  int64
NoOfAmpersandInURL              int64
NoOfOtherSpecialCharsInURL      int64
SpacialCharRatioInURL         float64
HasTitle                        int64
Title                          object
DomainTitleMatchScore         float64
URLTitleMatchScore            float64
NoOfURLRedirect                 int64
NoOfSelfRedirec

In [37]:
print("\nEmail:")
print(
    email_features[
        email_identity_candidates
    ].dtypes
)

print("\nPhiUSIIL:")
print(
    phiusiil_features[
        phiusiil_identity_candidates
    ].dtypes
)


Email:
sender                object
subject               object
urls                 float64
normalized_sender     object
dtype: object

PhiUSIIL:
URL                            object
URLLength                       int64
Domain                         object
DomainLength                    int64
IsDomainIP                      int64
URLSimilarityIndex            float64
URLCharProb                   float64
NoOfSubDomain                   int64
NoOfLettersInURL                int64
LetterRatioInURL              float64
NoOfDegitsInURL                 int64
DegitRatioInURL               float64
NoOfEqualsInURL                 int64
NoOfQMarkInURL                  int64
NoOfAmpersandInURL              int64
NoOfOtherSpecialCharsInURL      int64
SpacialCharRatioInURL         float64
HasTitle                        int64
Title                          object
DomainTitleMatchScore         float64
URLTitleMatchScore            float64
NoOfURLRedirect                 int64
NoOfSelfRedirec

### 7.1 Identity and Structural Evidence Quality Check

Identity-related indicators require stronger evidence than the presence of a
brand or authority-related word.

The available fields are therefore inspected programmatically before defining
identity-based extraction rules.

This prevents unsupported assumptions such as treating every sender-domain
difference as impersonation or inventing thresholds for webpage similarity
scores without first understanding their source representation.

In [38]:
email_identity_quality = {
    "sender_non_null": int(
        email_features["sender"].notna().sum()
    ),

    "sender_non_empty": int(
        (
            email_features["sender"]
            .fillna("")
            .astype(str)
            .str.strip()
            != ""
        ).sum()
    ),

    "urls_non_null": int(
        email_features["urls"].notna().sum()
    ),

    "urls_dtype": str(
        email_features["urls"].dtype
    )
}

email_identity_quality

{'sender_non_null': 22968,
 'sender_non_empty': 22968,
 'urls_non_null': 22968,
 'urls_dtype': 'float64'}

In [39]:
email_url_value_types = (
    email_features.loc[
        email_features["urls"].notna(),
        "urls"
    ]
    .map(lambda x: type(x).__name__)
    .value_counts()
)

email_url_value_types

urls
float    22968
Name: count, dtype: int64

In [40]:
phiusiil_identity_fields = [
    "DomainTitleMatchScore",
    "URLTitleMatchScore",
    "HasExternalFormSubmit",
    "HasSubmitButton",
    "HasPasswordField",
    "NoOfURLRedirect",
    "NoOfSelfRedirect",
    "NoOfiFrame",
    "NoOfExternalRef"
]

phiusiil_identity_quality = (
    phiusiil_features[
        phiusiil_identity_fields
    ]
    .agg(
        [
            "count",
            "min",
            "max",
            "mean"
        ]
    )
    .T
)

phiusiil_identity_quality

,count,min,max,mean
DomainTitleMatchScore,235795.0,0.0,100.0,50.131427
URLTitleMatchScore,235795.0,0.0,100.0,52.122098
HasExternalFormSubmit,235795.0,0.0,1.0,0.043987
HasSubmitButton,235795.0,0.0,1.0,0.414301
HasPasswordField,235795.0,0.0,1.0,0.102263
NoOfURLRedirect,235795.0,0.0,1.0,0.133438
NoOfSelfRedirect,235795.0,0.0,1.0,0.040107
NoOfiFrame,235795.0,0.0,1602.0,1.588638
NoOfExternalRef,235795.0,0.0,27516.0,49.262516


In [41]:
for col in [
    "HasExternalFormSubmit",
    "HasSubmitButton",
    "HasPasswordField"
]:
    print(
        col,
        sorted(
            phiusiil_features[col]
            .dropna()
            .unique()
            .tolist()
        )
    )

HasExternalFormSubmit [0, 1]
HasSubmitButton [0, 1]
HasPasswordField [0, 1]


In [42]:
email_identity_quality

{'sender_non_null': 22968,
 'sender_non_empty': 22968,
 'urls_non_null': 22968,
 'urls_dtype': 'float64'}

In [43]:
email_url_value_types

urls
float    22968
Name: count, dtype: int64

In [44]:
phiusiil_identity_quality

,count,min,max,mean
DomainTitleMatchScore,235795.0,0.0,100.0,50.131427
URLTitleMatchScore,235795.0,0.0,100.0,52.122098
HasExternalFormSubmit,235795.0,0.0,1.0,0.043987
HasSubmitButton,235795.0,0.0,1.0,0.414301
HasPasswordField,235795.0,0.0,1.0,0.102263
NoOfURLRedirect,235795.0,0.0,1.0,0.133438
NoOfSelfRedirect,235795.0,0.0,1.0,0.040107
NoOfiFrame,235795.0,0.0,1602.0,1.588638
NoOfExternalRef,235795.0,0.0,27516.0,49.262516


### 7.2 Identity-Indicator Availability Decision

The available evidence does not provide a sufficiently direct and
source-independent basis for automatically asserting impersonation or brand
exploitation.

For email observations, sender information is available for some source
datasets, but no verified claimed-identity field is available for comparison.
The processed `urls` field is also numeric rather than a usable raw URL string.

For PhiUSIIL, domain/title similarity scores are available, but no arbitrary
threshold is introduced to convert those continuous scores into impersonation
or brand-exploitation labels.

Accordingly, `impersonation` and `brand_exploitation` are marked unavailable
unless a later source-supported rule can be established.

This does not make the entire Trust / Identity dimension unavailable because
other indicators in the dimension, including Authority and
Confidentiality / Isolation, remain directly assessable.

In [45]:
for df in [
    email_features,
    phiusiil_features
]:
    df["impersonation_available"] = False
    df["impersonation_supported"] = False
    df["impersonation_evidence"] = "[]"

    df["brand_exploitation_available"] = False
    df["brand_exploitation_supported"] = False
    df["brand_exploitation_evidence"] = "[]"

## 8. Structural Webpage Evidence

PhiUSIIL exposes explicit binary webpage-structure fields that provide
deterministic evidence for requested-action behaviors.

The following source fields are used directly:

- `HasSubmitButton = 1` supports Call-to-Action;
- `HasExternalFormSubmit = 1` supports Call-to-Action; and
- `HasPasswordField = 1` supports Credential / Sensitive-Data Request.

Structural evidence supplements lexical evidence from the page title. An
indicator is supported when either its lexical rule or qualifying structural
evidence is present.

In [46]:
phiusiil_features["call_to_action_lexical_supported"] = (
    phiusiil_features["call_to_action_supported"]
)

phiusiil_features["credential_sensitive_request_lexical_supported"] = (
    phiusiil_features["credential_sensitive_request_supported"]
)

In [47]:
def webpage_structural_evidence(row):
    evidence = []

    if row["HasSubmitButton"] == 1:
        evidence.append("HasSubmitButton=1")

    if row["HasExternalFormSubmit"] == 1:
        evidence.append("HasExternalFormSubmit=1")

    return evidence


phiusiil_features["call_to_action_structural_evidence"] = (
    phiusiil_features.apply(
        webpage_structural_evidence,
        axis=1
    )
)

phiusiil_features["credential_structural_evidence"] = (
    phiusiil_features["HasPasswordField"]
    .map(
        lambda x:
            ["HasPasswordField=1"]
            if x == 1
            else []
    )
)

In [48]:
def combine_json_and_list(json_evidence, structural_evidence):
    lexical = json.loads(json_evidence)

    combined = lexical + structural_evidence

    # Preserve order and remove duplicates
    return list(dict.fromkeys(combined))


phiusiil_features["call_to_action_evidence"] = [
    json.dumps(
        combine_json_and_list(
            lexical,
            structural
        ),
        ensure_ascii=False
    )
    for lexical, structural in zip(
        phiusiil_features["call_to_action_evidence"],
        phiusiil_features["call_to_action_structural_evidence"]
    )
]

phiusiil_features["credential_sensitive_request_evidence"] = [
    json.dumps(
        combine_json_and_list(
            lexical,
            structural
        ),
        ensure_ascii=False
    )
    for lexical, structural in zip(
        phiusiil_features["credential_sensitive_request_evidence"],
        phiusiil_features["credential_structural_evidence"]
    )
]

In [49]:
phiusiil_features["call_to_action_supported"] = (
    phiusiil_features[
        "call_to_action_lexical_supported"
    ]
    |
    phiusiil_features[
        "call_to_action_structural_evidence"
    ].map(bool)
)

phiusiil_features[
    "credential_sensitive_request_supported"
] = (
    phiusiil_features[
        "credential_sensitive_request_lexical_supported"
    ]
    |
    phiusiil_features[
        "credential_structural_evidence"
    ].map(bool)
)

In [50]:
print(
    "Call-to-Action before structural evidence:",
    phiusiil_features[
        "call_to_action_lexical_supported"
    ].sum()
)

print(
    "Call-to-Action after structural evidence:",
    phiusiil_features[
        "call_to_action_supported"
    ].sum()
)

print()

print(
    "Credential request before structural evidence:",
    phiusiil_features[
        "credential_sensitive_request_lexical_supported"
    ].sum()
)

print(
    "Credential request after structural evidence:",
    phiusiil_features[
        "credential_sensitive_request_supported"
    ].sum()
)

Call-to-Action before structural evidence: 371
Call-to-Action after structural evidence: 98946

Credential request before structural evidence: 8
Credential request after structural evidence: 24121


## 9. SE-BRL Dimension-State Construction

Indicator-level evidence is aggregated into the four SE-BRL behavioral
dimensions.

Each dimension receives one of three states:

- `Supported` — at least one available indicator in the dimension is supported;
- `Absent` — the dimension is assessable, but none of its available indicators
  is supported; or
- `Unavailable` — none of the indicators required to assess the dimension is
  available for the observation.

Unavailable indicators do not count as negative evidence.

This distinction ensures that missing modality evidence is not incorrectly
interpreted as behavioral absence.

In [51]:
email_text_available = (
    email_features["evidence_text"]
    .fillna("")
    .str.strip()
    .ne("")
)

for indicator_id in LEXICAL_INDICATORS:
    email_features[
        f"{indicator_id}_available"
    ] = email_text_available

# Already established as unavailable
email_features["impersonation_available"] = False
email_features["brand_exploitation_available"] = False

In [52]:
phiusiil_title_available = (
    phiusiil_features["evidence_text"]
    .fillna("")
    .str.strip()
    .ne("")
)

for indicator_id in LEXICAL_INDICATORS:
    phiusiil_features[
        f"{indicator_id}_available"
    ] = phiusiil_title_available

# Structural webpage fields make these assessable for every row
phiusiil_features["call_to_action_available"] = True
phiusiil_features[
    "credential_sensitive_request_available"
] = True

# No source-supported rule established
phiusiil_features["impersonation_available"] = False
phiusiil_features["brand_exploitation_available"] = False

In [53]:
for df in [
    uci_features,
    iscx_features
]:
    for indicator_id in expected_indicators:

        df[
            f"{indicator_id}_available"
        ] = False

        df[
            f"{indicator_id}_supported"
        ] = False

        df[
            f"{indicator_id}_evidence"
        ] = "[]"

In [54]:
def derive_dimension_state(
    row,
    indicator_ids
):
    available = [
        bool(
            row[
                f"{indicator_id}_available"
            ]
        )
        for indicator_id in indicator_ids
    ]

    supported = [
        bool(
            row[
                f"{indicator_id}_supported"
            ]
        )
        for indicator_id in indicator_ids
    ]

    # Nothing in this dimension can be assessed
    if not any(available):
        return "Unavailable"

    # At least one available indicator has evidence
    if any(
        is_available and is_supported
        for is_available, is_supported
        in zip(available, supported)
    ):
        return "Supported"

    return "Absent"

In [55]:
DIMENSION_INDICATORS = {
    dimension_id: list(
        dimension_info["indicators"].keys()
    )
    for dimension_id, dimension_info
    in SEBRL_CODEBOOK.items()
}

DIMENSION_INDICATORS

{'pressure_threat': ['urgency', 'fear_threat', 'scarcity'],
 'lure_attention': ['curiosity', 'reward_lure'],
 'trust_identity': ['authority',
  'impersonation',
  'brand_exploitation',
  'confidentiality_isolation'],
 'requested_action_consequence': ['call_to_action',
  'credential_sensitive_request',
  'financial_action_request']}

In [56]:
for df in [
    email_features,
    phiusiil_features,
    uci_features,
    iscx_features
]:

    for dimension_id, indicator_ids in (
        DIMENSION_INDICATORS.items()
    ):

        df[
            f"{dimension_id}_state"
        ] = df.apply(
            lambda row:
                derive_dimension_state(
                    row,
                    indicator_ids
                ),
            axis=1
        )

In [57]:
dimension_state_summary = []

for dataset_name, df in {
    "Email": email_features,
    "PhiUSIIL": phiusiil_features,
    "UCI": uci_features,
    "ISCX": iscx_features
}.items():

    for dimension_id in DIMENSION_INDICATORS:

        counts = (
            df[
                f"{dimension_id}_state"
            ]
            .value_counts()
        )

        dimension_state_summary.append({
            "dataset": dataset_name,
            "dimension": dimension_id,
            "supported": int(
                counts.get("Supported", 0)
            ),
            "absent": int(
                counts.get("Absent", 0)
            ),
            "unavailable": int(
                counts.get("Unavailable", 0)
            )
        })

dimension_state_summary = pd.DataFrame(
    dimension_state_summary
)

dimension_state_summary

,dataset,dimension,supported,absent,unavailable
0,Email,pressure_threat,3977,34782,0
1,Email,lure_attention,5413,33346,0
2,Email,trust_identity,6431,32328,0
3,Email,requested_action_consequence,13771,24988,0
4,PhiUSIIL,pressure_threat,29,235766,0
5,PhiUSIIL,lure_attention,67,235728,0
6,PhiUSIIL,trust_identity,104,235691,0
7,PhiUSIIL,requested_action_consequence,101911,133884,0
8,UCI,pressure_threat,0,0,5849
9,UCI,lure_attention,0,0,5849


## 10. SE-BRL Behavioral Vector and Availability Mask

The categorical dimension states are converted into a numeric representation
for downstream machine-learning experiments.

For each SE-BRL dimension, two values are retained:

1. a behavioral support value:
   - `1` = Supported
   - `0` = Absent or Unavailable

2. an availability mask:
   - `1` = the dimension is assessable
   - `0` = the dimension is Unavailable

The availability mask prevents unavailable evidence from being interpreted as
behavioral absence.

For example:

- `support = 1, availability = 1` means Supported;
- `support = 0, availability = 1` means Absent; and
- `support = 0, availability = 0` means Unavailable.

This two-part representation preserves modality awareness while remaining
compatible with standard machine-learning models.

In [58]:
def encode_dimension_state(state):
    if state == "Supported":
        return 1, 1

    if state == "Absent":
        return 0, 1

    if state == "Unavailable":
        return 0, 0

    raise ValueError(
        f"Unexpected SE-BRL state: {state}"
    )

In [59]:
sebrl_datasets = {
    "Email": email_features,
    "PhiUSIIL": phiusiil_features,
    "UCI": uci_features,
    "ISCX": iscx_features
}

for dataset_name, df in sebrl_datasets.items():

    for dimension_id in DIMENSION_INDICATORS:

        state_col = f"{dimension_id}_state"

        encoded = df[state_col].map(
            encode_dimension_state
        )

        df[f"{dimension_id}_supported"] = (
            encoded.map(lambda x: x[0])
        )

        df[f"{dimension_id}_available"] = (
            encoded.map(lambda x: x[1])
        )

In [60]:
SEBRL_SUPPORT_COLUMNS = [
    f"{dimension_id}_supported"
    for dimension_id in DIMENSION_INDICATORS
]

SEBRL_AVAILABILITY_COLUMNS = [
    f"{dimension_id}_available"
    for dimension_id in DIMENSION_INDICATORS
]

print("Support vector:")
print(SEBRL_SUPPORT_COLUMNS)

print("\nAvailability mask:")
print(SEBRL_AVAILABILITY_COLUMNS)

Support vector:
['pressure_threat_supported', 'lure_attention_supported', 'trust_identity_supported', 'requested_action_consequence_supported']

Availability mask:
['pressure_threat_available', 'lure_attention_available', 'trust_identity_available', 'requested_action_consequence_available']


In [61]:
for dataset_name, df in sebrl_datasets.items():

    print(f"\n{dataset_name}")

    for dimension_id in DIMENSION_INDICATORS:

        state_col = f"{dimension_id}_state"
        support_col = f"{dimension_id}_supported"
        available_col = f"{dimension_id}_available"

        check = pd.crosstab(
            df[state_col],
            [
                df[support_col],
                df[available_col]
            ]
        )

        print(f"\n{dimension_id}")
        print(check)


Email

pressure_threat
pressure_threat_supported      0     1
pressure_threat_available      1     1
pressure_threat_state                 
Absent                     34782     0
Supported                      0  3977

lure_attention
lure_attention_supported      0     1
lure_attention_available      1     1
lure_attention_state                 
Absent                    33346     0
Supported                     0  5413

trust_identity
trust_identity_supported      0     1
trust_identity_available      1     1
trust_identity_state                 
Absent                    32328     0
Supported                     0  6431

requested_action_consequence
requested_action_consequence_supported      0      1
requested_action_consequence_available      1      1
requested_action_consequence_state                  
Absent                                  24988      0
Supported                                   0  13771

PhiUSIIL

pressure_threat
pressure_threat_supported       0   1
pressure_

In [62]:
for dataset_name, df in sebrl_datasets.items():

    for dimension_id in DIMENSION_INDICATORS:

        state_col = f"{dimension_id}_state"
        support_col = f"{dimension_id}_supported"
        available_col = f"{dimension_id}_available"

        assert (
            df.loc[
                df[state_col] == "Supported",
                support_col
            ]
            == 1
        ).all()

        assert (
            df.loc[
                df[state_col] == "Supported",
                available_col
            ]
            == 1
        ).all()

        assert (
            df.loc[
                df[state_col] == "Absent",
                support_col
            ]
            == 0
        ).all()

        assert (
            df.loc[
                df[state_col] == "Absent",
                available_col
            ]
            == 1
        ).all()

        assert (
            df.loc[
                df[state_col] == "Unavailable",
                support_col
            ]
            == 0
        ).all()

        assert (
            df.loc[
                df[state_col] == "Unavailable",
                available_col
            ]
            == 0
        ).all()

print(
    "All SE-BRL dimension states encoded correctly."
)

All SE-BRL dimension states encoded correctly.


In [63]:
vector_summary = []

for dataset_name, df in sebrl_datasets.items():

    for dimension_id in DIMENSION_INDICATORS:

        vector_summary.append({
            "dataset": dataset_name,
            "dimension": dimension_id,
            "support_rate_percent": round(
                df[
                    f"{dimension_id}_supported"
                ].mean() * 100,
                2
            ),
            "availability_rate_percent": round(
                df[
                    f"{dimension_id}_available"
                ].mean() * 100,
                2
            )
        })

vector_summary = pd.DataFrame(
    vector_summary
)

vector_summary

,dataset,dimension,support_rate_percent,availability_rate_percent
0,Email,pressure_threat,10.26,100.0
1,Email,lure_attention,13.97,100.0
2,Email,trust_identity,16.59,100.0
3,Email,requested_action_consequence,35.53,100.0
4,PhiUSIIL,pressure_threat,0.01,100.0
5,PhiUSIIL,lure_attention,0.03,100.0
6,PhiUSIIL,trust_identity,0.04,100.0
7,PhiUSIIL,requested_action_consequence,43.22,100.0
8,UCI,pressure_threat,0.00,0.0
9,UCI,lure_attention,0.00,0.0


## 11. Indicator-Level Evidence Representation

The compact SE-BRL model vector represents behavioral support at the
four-dimension level.

Indicator-level outputs are retained separately for explainability and
validation.

For each of the twelve indicators, the exported evidence representation
contains:

- indicator availability;
- indicator support status; and
- exact automatically extracted evidence.

Keeping the indicator evidence separate from the compact model vector allows
downstream validation of evidence localization without introducing evidence
strings directly into the classifier input.

In [64]:
SEBRL_INDICATORS = (
    sebrl_codebook_df["indicator_id"]
    .tolist()
)

print("Indicators:", len(SEBRL_INDICATORS))
print(SEBRL_INDICATORS)

Indicators: 12
['urgency', 'fear_threat', 'scarcity', 'curiosity', 'reward_lure', 'authority', 'impersonation', 'brand_exploitation', 'confidentiality_isolation', 'call_to_action', 'credential_sensitive_request', 'financial_action_request']


In [66]:
def build_indicator_evidence_table(df, dataset_name):
    rows = []

    for indicator_id in SEBRL_INDICATORS:

        availability_col = f"{indicator_id}_available"
        support_col = f"{indicator_id}_supported"
        evidence_col = f"{indicator_id}_evidence"

        temp = pd.DataFrame({
            "observation_id": df["observation_id"],
            "dataset": dataset_name,
            "indicator_id": indicator_id,
            "available": df[availability_col],
            "supported": df[support_col],
            "evidence": df[evidence_col],
        })

        rows.append(temp)

    return pd.concat(
        rows,
        ignore_index=True
    )

In [67]:
email_indicator_evidence = (
    build_indicator_evidence_table(
        email_features,
        "Email"
    )
)

phiusiil_indicator_evidence = (
    build_indicator_evidence_table(
        phiusiil_features,
        "PhiUSIIL"
    )
)

uci_indicator_evidence = (
    build_indicator_evidence_table(
        uci_features,
        "UCI"
    )
)

iscx_indicator_evidence = (
    build_indicator_evidence_table(
        iscx_features,
        "ISCX"
    )
)

In [68]:
indicator_evidence = pd.concat(
    [
        email_indicator_evidence,
        phiusiil_indicator_evidence,
        uci_indicator_evidence,
        iscx_indicator_evidence,
    ],
    ignore_index=True
)

print(
    "Indicator evidence rows:",
    len(indicator_evidence)
)

indicator_evidence.head()

Indicator evidence rows: 3542712


,observation_id,dataset,indicator_id,available,supported,evidence
0,Nazario::0,Email,urgency,True,True,"[""URGENT"", ""immediately"", ""ASAP"", ""within 24 hours"", ""within 48 hours"", ""deadline"", ""expire""]"
1,Nazario::1,Email,urgency,True,False,[]
2,Nazario::2,Email,urgency,True,False,[]
3,Nazario::3,Email,urgency,True,True,"[""expire""]"
4,Nazario::4,Email,urgency,True,True,"[""Deadline""]"


In [69]:
invalid_unavailable_support = (
    indicator_evidence[
        (~indicator_evidence["available"])
        &
        (indicator_evidence["supported"])
    ]
)

print(
    "Unavailable-but-supported records:",
    len(invalid_unavailable_support)
)

Unavailable-but-supported records: 0


In [70]:
indicator_evidence_counts = (
    indicator_evidence
    .groupby("dataset")
    .size()
)

indicator_evidence_counts

dataset
Email        465108
ISCX         177876
PhiUSIIL    2829540
UCI           70188
dtype: int64

In [71]:
assert len(indicator_evidence) == (
    (
        len(email_features)
        + len(phiusiil_features)
        + len(uci_features)
        + len(iscx_features)
    )
    * 12
)

assert len(invalid_unavailable_support) == 0

print(
    "All indicator-level evidence integrity checks passed."
)

All indicator-level evidence integrity checks passed.


## 12. Compact SE-BRL Model Feature Tables

The final SE-BRL model representation contains only the metadata required to
preserve the experimental design together with the compact behavioral vector.

For each observation, the exported model table contains:

- observation identifier;
- target;
- leakage-prevention group identifier;
- development/calibration/test partition;
- development cross-validation fold;
- four SE-BRL dimension support values; and
- four modality-availability mask values.

Indicator evidence strings remain stored separately and are not included as
direct classifier inputs.

In [72]:
BASE_MODEL_COLUMNS = [
    "observation_id",
    "target",
    "group_id",
    "partition",
    "cv_fold",
]

SEBRL_MODEL_COLUMNS = (
    BASE_MODEL_COLUMNS
    + SEBRL_SUPPORT_COLUMNS
    + SEBRL_AVAILABILITY_COLUMNS
)

SEBRL_MODEL_COLUMNS

['observation_id',
 'target',
 'group_id',
 'partition',
 'cv_fold',
 'pressure_threat_supported',
 'lure_attention_supported',
 'trust_identity_supported',
 'requested_action_consequence_supported',
 'pressure_threat_available',
 'lure_attention_available',
 'trust_identity_available',
 'requested_action_consequence_available']

In [73]:
email_sebrl = (
    email_features[
        SEBRL_MODEL_COLUMNS
    ]
    .copy()
)

phiusiil_sebrl = (
    phiusiil_features[
        SEBRL_MODEL_COLUMNS
    ]
    .copy()
)

uci_sebrl = (
    uci_features[
        SEBRL_MODEL_COLUMNS
    ]
    .copy()
)

iscx_sebrl = (
    iscx_features[
        SEBRL_MODEL_COLUMNS
    ]
    .copy()
)

In [74]:
for name, df in {
    "Email": email_sebrl,
    "PhiUSIIL": phiusiil_sebrl,
    "UCI": uci_sebrl,
    "ISCX": iscx_sebrl,
}.items():

    print(
        f"{name}: "
        f"{len(df):,} rows × {len(df.columns)} columns"
    )

Email: 38,759 rows × 13 columns
PhiUSIIL: 235,795 rows × 13 columns
UCI: 5,849 rows × 13 columns
ISCX: 14,823 rows × 13 columns


In [75]:
for name, df in {
    "Email": email_sebrl,
    "PhiUSIIL": phiusiil_sebrl,
    "UCI": uci_sebrl,
    "ISCX": iscx_sebrl,
}.items():

    assert df["observation_id"].is_unique
    assert df["partition"].notna().all()

    assert (
        df.loc[
            df["partition"] == "development",
            "cv_fold"
        ]
        .notna()
        .all()
    )

    assert (
        df.loc[
            df["partition"] != "development",
            "cv_fold"
        ]
        .isna()
        .all()
    )

print(
    "All compact SE-BRL feature tables "
    "preserve preprocessing metadata."
)

All compact SE-BRL feature tables preserve preprocessing metadata.


In [76]:
for name, df in {
    "Email": email_sebrl,
    "PhiUSIIL": phiusiil_sebrl,
    "UCI": uci_sebrl,
    "ISCX": iscx_sebrl,
}.items():

    print(f"\n{name}")

    print(
        "Support values:",
        sorted(
            pd.unique(
                df[
                    SEBRL_SUPPORT_COLUMNS
                ].values.ravel()
            )
        )
    )

    print(
        "Availability values:",
        sorted(
            pd.unique(
                df[
                    SEBRL_AVAILABILITY_COLUMNS
                ].values.ravel()
            )
        )
    )


Email
Support values: [np.int64(0), np.int64(1)]
Availability values: [np.int64(1)]

PhiUSIIL
Support values: [np.int64(0), np.int64(1)]
Availability values: [np.int64(1)]

UCI
Support values: [np.int64(0)]
Availability values: [np.int64(0)]

ISCX
Support values: [np.int64(0)]
Availability values: [np.int64(0)]


## 13. Export SE-BRL Feature Artifacts

The finalized SE-BRL outputs are exported in separate representations for
modeling, validation, and documentation.

The compact model tables contain only the four dimension-level support values
and four availability-mask values together with immutable experimental
metadata.

Indicator-level evidence is exported separately so that explanation and
validation can be performed without exposing evidence strings directly to the
classifiers.

The canonical SE-BRL codebook and summary tables are also exported for
reproducibility.

In [77]:
SEBRL_FEATURE_DIR = (
    FEATURE_DATA_DIR
    / "sebrl"
)

SEBRL_EVIDENCE_DIR = (
    FEATURE_RESULTS_DIR
    / "evidence"
)

SEBRL_SUMMARY_DIR = (
    FEATURE_RESULTS_DIR
    / "summaries"
)

SEBRL_FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SEBRL_EVIDENCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SEBRL_SUMMARY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [78]:
sebrl_exports = {
    "email_sebrl_features.csv":
        email_sebrl,

    "phiusiil_sebrl_features.csv":
        phiusiil_sebrl,

    "uci_sebrl_features.csv":
        uci_sebrl,

    "iscx_sebrl_features.csv":
        iscx_sebrl,
}

for filename, df in sebrl_exports.items():

    output_path = (
        SEBRL_FEATURE_DIR
        / filename
    )

    df.to_csv(
        output_path,
        index=False
    )

    print(
        f"{filename}: "
        f"{len(df):,} rows × "
        f"{len(df.columns)} columns"
    )

email_sebrl_features.csv: 38,759 rows × 13 columns
phiusiil_sebrl_features.csv: 235,795 rows × 13 columns
uci_sebrl_features.csv: 5,849 rows × 13 columns
iscx_sebrl_features.csv: 14,823 rows × 13 columns


In [79]:
indicator_evidence_path = (
    SEBRL_EVIDENCE_DIR
    / "indicator_evidence.csv"
)

indicator_evidence.to_csv(
    indicator_evidence_path,
    index=False
)

print(
    "Indicator evidence:",
    f"{len(indicator_evidence):,} rows"
)

Indicator evidence: 3,542,712 rows


In [80]:
sebrl_codebook_df.to_csv(
    FEATURE_RESULTS_DIR
    / "sebrl_codebook.csv",
    index=False
)

evidence_availability_df.to_csv(
    FEATURE_RESULTS_DIR
    / "evidence_availability.csv",
    index=False
)

lexical_support_summary.to_csv(
    SEBRL_SUMMARY_DIR
    / "lexical_support_summary.csv",
    index=False
)

dimension_state_summary.to_csv(
    SEBRL_SUMMARY_DIR
    / "dimension_state_summary.csv",
    index=False
)

vector_summary.to_csv(
    SEBRL_SUMMARY_DIR
    / "vector_summary.csv",
    index=False
)

In [81]:
expected_feature_outputs = [
    SEBRL_FEATURE_DIR
    / "email_sebrl_features.csv",

    SEBRL_FEATURE_DIR
    / "phiusiil_sebrl_features.csv",

    SEBRL_FEATURE_DIR
    / "uci_sebrl_features.csv",

    SEBRL_FEATURE_DIR
    / "iscx_sebrl_features.csv",

    SEBRL_EVIDENCE_DIR
    / "indicator_evidence.csv",

    FEATURE_RESULTS_DIR
    / "sebrl_codebook.csv",

    FEATURE_RESULTS_DIR
    / "evidence_availability.csv",

    SEBRL_SUMMARY_DIR
    / "lexical_support_summary.csv",

    SEBRL_SUMMARY_DIR
    / "dimension_state_summary.csv",

    SEBRL_SUMMARY_DIR
    / "vector_summary.csv",
]

for path in expected_feature_outputs:

    print(
        f"{'OK' if path.exists() else 'MISSING'} "
        f"{path.relative_to(REPO_ROOT)}"
    )

OK experiments/data/processed/features/sebrl/email_sebrl_features.csv
OK experiments/data/processed/features/sebrl/phiusiil_sebrl_features.csv
OK experiments/data/processed/features/sebrl/uci_sebrl_features.csv
OK experiments/data/processed/features/sebrl/iscx_sebrl_features.csv
OK experiments/results/feature_extraction/evidence/indicator_evidence.csv
OK experiments/results/feature_extraction/sebrl_codebook.csv
OK experiments/results/feature_extraction/evidence_availability.csv
OK experiments/results/feature_extraction/summaries/lexical_support_summary.csv
OK experiments/results/feature_extraction/summaries/dimension_state_summary.csv
OK experiments/results/feature_extraction/summaries/vector_summary.csv


## 14. Feature Extraction Summary and Documentation

### Completed Feature Extraction Tasks

The SE-BRL feature-extraction stage produced a deterministic,
modality-aware behavioral representation from the fixed preprocessing outputs
generated by `03_preprocessing.ipynb`.

The following tasks were completed:

- loaded the fixed preprocessing datasets and verified immutable partition
  metadata;
- defined the canonical four-dimension, twelve-indicator SE-BRL codebook;
- established modality-specific evidence availability rules;
- implemented deterministic lexical indicator extraction;
- preserved exact automatically matched evidence for explainability;
- incorporated deterministic webpage structural evidence from PhiUSIIL;
- explicitly separated unsupported identity inference from unavailable
  evidence;
- constructed dimension-level `Supported`, `Absent`, and `Unavailable` states;
- encoded the four-dimensional SE-BRL support vector and modality availability
  mask;
- constructed indicator-level evidence records for downstream validation; and
- exported compact SE-BRL model features and supporting documentation artifacts.

### Canonical SE-BRL Representation

The behavioral representation contains four dimensions:

1. Pressure / Threat
2. Lure / Attention
3. Trust / Identity
4. Requested Action / Consequence

These dimensions are derived from twelve subordinate indicators.

Each dimension is represented using:

- a support value (`1` = Supported, `0` otherwise); and
- an availability value (`1` = assessable, `0` = Unavailable).

This preserves the distinction between behavioral absence and unavailable
modality evidence.

### Email Feature Extraction

Email observations use available subject and body content for lexical
behavioral extraction.

The following dimension support rates were observed:

- Pressure / Threat: approximately 10.26%;
- Lure / Attention: approximately 13.97%;
- Trust / Identity: approximately 16.59%; and
- Requested Action / Consequence: approximately 35.53%.

The email modality remains fully assessable for the four dimension-level
representations under the current extraction rules.

### PhiUSIIL Feature Extraction

PhiUSIIL lexical extraction uses the available webpage title field.

Explicit webpage-structure fields were additionally used as deterministic
evidence:

- `HasSubmitButton = 1`;
- `HasExternalFormSubmit = 1`; and
- `HasPasswordField = 1`.

These structural fields substantially increased detection of requested-action
behaviors compared with title-only lexical extraction.

The final Requested Action / Consequence support rate is approximately 43.22%.

No arbitrary threshold was introduced for continuous domain/title similarity
scores.

### Identity-Based Indicators

`Impersonation` and `Brand Exploitation` were not inferred using arbitrary
sender or domain mismatch rules.

The available datasets do not provide a sufficiently direct and
source-independent basis for asserting these indicators automatically under the
current extraction design.

They are therefore treated as unavailable where a defensible deterministic
rule cannot be established.

Other indicators within the Trust / Identity dimension remain assessable.

### URL-Only Datasets

UCI and ISCX contain engineered URL or webpage-characteristic features without
the underlying natural-language or page-content evidence required for direct
SE-BRL behavioral inference.

Accordingly, their behavioral dimensions are encoded as:

- support = `0`; and
- availability = `0`.

This represents `Unavailable`, not behavioral absence.

### Evidence Representation

Indicator-level evidence is stored separately from the compact classifier
inputs.

Each indicator evidence record preserves:

- observation identifier;
- dataset;
- indicator identifier;
- availability;
- support status; and
- automatically extracted evidence.

The combined indicator evidence table contains 3,542,712 records.

No unavailable indicator was simultaneously marked as supported.

### Generated Artifacts

Compact SE-BRL model feature tables:

`experiments/data/processed/features/sebrl/`

Indicator-level evidence:

`experiments/results/feature_extraction/evidence/`

Canonical codebook and evidence-availability documentation:

`experiments/results/feature_extraction/`

Feature-extraction summary tables:

`experiments/results/feature_extraction/summaries/`

### Next Stage

The next stage is `05_experiments.ipynb`.

That notebook will use the fixed development, calibration, test, and
cross-validation assignments together with the SE-BRL feature representations
to execute the controlled machine-learning experiments.

Learned preprocessing, classifier fitting, threshold calibration, and final
test evaluation must continue to respect the leakage controls established in
the preceding stages.